# Stage 3 -- Materialization

Writes every sample from the merged parquets as an individual source file suitable for static analysis tools.

**Output layout:**
```
data/materialized/
  <lang>/
    <dataset>/
      <sample_id>.<ext>   # one file per sample
    index.parquet         # lookup: sample_id -> source, label, cwes, branch, code_hash
```

**Extensions:** `.c` for C/C++, `.java` for Java, `.py` for Python.

> **Note (Java):** Extracted samples are function bodies, not compilable top-level classes.
> Tools that require a full compilation unit will need a wrapping step.

**Prerequisites:** run `02_synthesis.ipynb` first to populate `data/merged/`.

In [ ]:
import sys
import re
from pathlib import Path

import pandas as pd

ROOT       = Path('..').resolve()
MERGED_DIR = ROOT / 'data' / 'merged'
MAT_DIR    = ROOT / 'data' / 'materialized'

_LANG_SLUG = {'C/C++': 'c_cpp', 'Java': 'java', 'Python': 'python'}
_LANG_EXT  = {'C/C++': '.c',    'Java': '.java', 'Python': '.py'}


def _slugify(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')


print(f'Root:          {ROOT}')
print(f'Merged:        {MERGED_DIR}')
print(f'Materialized:  {MAT_DIR}')

## 1. Write source files

In [ ]:
OVERWRITE = False  # set True to re-write files that already exist

total_written = 0
total_skipped = 0

for language, slug in _LANG_SLUG.items():
    ext   = _LANG_EXT[language]
    parq  = MERGED_DIR / f'{slug}_merged.parquet'

    if not parq.exists():
        print(f'[{language}] SKIP -- {parq.name} not found (run 02_synthesis.ipynb first)')
        continue

    df = pd.read_parquet(parq)
    if 'sample_id' not in df.columns:
        df['sample_id'] = df['code_hash'].str[:16]

    lang_dir  = MAT_DIR / slug
    n_written = 0
    n_skip    = 0

    for _, row in df.iterrows():
        dest = lang_dir / _slugify(row['source']) / f'{row["sample_id"]}{ext}'
        if not OVERWRITE and dest.exists():
            n_skip += 1
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(row['code'], encoding='utf-8')
        n_written += 1

    # per-language index
    index_path = lang_dir / 'index.parquet'
    lang_dir.mkdir(parents=True, exist_ok=True)
    df[['sample_id', 'source', 'label', 'cwes', 'branch', 'code_hash']].to_parquet(index_path, index=False)

    print(f'[{language}]  written={n_written:,}  skipped={n_skip:,}  index -> {index_path.relative_to(ROOT)}')
    total_written += n_written
    total_skipped += n_skip

print(f'\nTotal written : {total_written:,}')
print(f'Total skipped : {total_skipped:,}')
print(f'Output root   : {MAT_DIR.relative_to(ROOT)}')

## 2. Verify output structure

In [ ]:
print('Materialized layout:')
for lang_dir in sorted(MAT_DIR.iterdir()):
    if not lang_dir.is_dir():
        continue
    datasets = sorted(d for d in lang_dir.iterdir() if d.is_dir())
    total = sum(len(list(d.iterdir())) for d in datasets)
    print(f'  {lang_dir.name}/  ({total:,} files across {len(datasets)} datasets)')
    for d in datasets:
        n = len(list(d.iterdir()))
        print(f'    {d.name}/  {n:,}')
    index = lang_dir / 'index.parquet'
    if index.exists():
        idx = pd.read_parquet(index)
        n_vuln = int((idx['label'] == 1).sum())
        n_safe = int((idx['label'] == 0).sum())
        print(f'    index.parquet  {len(idx):,} rows  ({n_vuln:,} vuln + {n_safe:,} safe)')